# My Own Analysis Project

## week 1


### 🛠️ Week 1: Data Alignment & Master Feature Table Construction
* **목표:** 기존 실습 환경에 준비된 전처리 데이터(`read-count.txt` 및 CLIP-seq 파일)를 로드하고, 이를 유전자 서열 데이터와 결합하여 머신러닝 학습을 위한 통합 마스터 테이블(Master Dataframe)을 구축합니다.
* **주요 세부 과제:**
  1. **로컬 데이터 로드:** `read-count.txt` 파일을 Pandas로 읽어 들여 유전자별 RNA-seq 및 Ribo-seq 카운트 데이터 구조 파악
  2. **번역 효율(TE) 산출:** 전사량(RNA) 대비 실제 번역량(Ribo)의 비율을 계산하여 LIN28A Knockdown 시 번역 변화량($\Delta$TE) 도출
  3. **면역/세포막 타겟 라벨링:** 유전자 이름(Gene Symbol) 매칭 혹은 키워드 필터링을 통해 면역(Immune) 및 세포막(Membrane) 유전자군을 분류하고 목적 변수(`is_immune`, `is_target`) 생성
  4. **서열 기반 피처 엔지니어링:** `lin28a-clip-seq.pileup` 데이터와 유전자 서열을 연동하여, 유전자별 기초 서열 특성(GC 비율, `GGAG` 결합 모티프 개수)을 계산하고 표에 결합
* **Commit Artifacts:** `data_preprocessing.ipynb`, `processed_master_table.csv`

##### 데이터 구조 파악

In [5]:
import pandas as pd
from pathlib import Path

# featureCounts 출력 (파일명: read-counts.txt, 첫 줄은 # 주석)
counts_path = Path('binfo1-work/read-counts.txt')
df = pd.read_csv(counts_path, sep='\t', comment='#')

# 데이터 상위 5줄 구경하기
df.head()

,Geneid,Chr,Start,End,Strand,Length,CLIP-35L33G.bam,CLIP-let7g.bam,RNA-control.bam,RNA-siLin28a.bam,RNA-siLuc.bam,RPF-siLin28a.bam,RPF-siLuc.bam
0,ENSMUSG00000102693.2,chr1,3143476,3144545,+,1070,0,0,0,0,0,0,0
1,ENSMUSG00000064842.3,chr1,3172239,3172348,+,110,0,0,0,0,0,0,0
2,ENSMUSG00000051951.6,chr1;chr1;chr1;chr1;chr1;chr1;chr1,3276124;3276746;3283662;3283832;3284705;349192...,3277540;3277540;3285855;3286567;3287191;349212...,-;-;-;-;-;-;-,6094,4,0,1,1,1,0,0
3,ENSMUSG00000102851.2,chr1,3322980,3323459,+,480,3,0,0,0,0,0,0
4,ENSMUSG00000103377.2,chr1,3435954,3438772,-,2819,0,0,0,0,0,0,0


##### 번역효율(TE)와 변화량(delta TE) 계산

In [7]:
import pandas as pd
import numpy as np

# 1. 데이터 다시 깔끔하게 불러오기 (첫 번째 줄이 주석일 수 있으니 skiprows 옵션 추가)
# 만약 에러가 나면 skiprows=1을 지우고 해보세요.
df = pd.read_csv('binfo1-work/read-counts.txt', sep='\t', comment='#')

# 2. 분석에 필요한 핵심 컬럼만 쏙 골라내기
cols_to_keep = ['Geneid', 'Length', 'RNA-siLuc.bam', 'RNA-siLin28a.bam', 'RPF-siLuc.bam', 'RPF-siLin28a.bam']
df_clean = df[cols_to_keep].copy()

# 3. 발현량이 너무 적은 '유령 유전자'들 필터링 (노이즈 제거)
# RNA 대조군에서 카운트가 10 이상인 유전자만 남깁니다.
df_clean = df_clean[df_clean['RNA-siLuc.bam'] >= 10]

# 4. 번역 효율(Translation Efficiency, TE) 계산! 
# (분모가 0이 되는 것을 막기 위해 모든 값에 +1이라는 Pseudo-count를 줍니다)
df_clean['TE_control'] = (df_clean['RPF-siLuc.bam'] + 1) / (df_clean['RNA-siLuc.bam'] + 1)
df_clean['TE_knockdown'] = (df_clean['RPF-siLin28a.bam'] + 1) / (df_clean['RNA-siLin28a.bam'] + 1)

# 5. Lin28a를 껐을 때 번역 효율이 얼마나 변했나? (Log2 Fold Change)
# 값이 양수(+)면 Lin28a가 없어졌을 때 번역이 떡상한 유전자 = Lin28a의 억제 타겟!
df_clean['log2_FC_TE'] = np.log2(df_clean['TE_knockdown'] / df_clean['TE_control'])

# 번역이 가장 많이 떡상한(증가한) 상위 5개 유전자 구경하기
print("🔥 Lin28a Knockdown 후 번역 효율이 가장 많이 증가한 Top 5 유전자 🔥")
display(df_clean.sort_values('log2_FC_TE', ascending=False).head())

🔥 Lin28a Knockdown 후 번역 효율이 가장 많이 증가한 Top 5 유전자 🔥


,Geneid,Length,RNA-siLuc.bam,RNA-siLin28a.bam,RPF-siLuc.bam,RPF-siLin28a.bam,TE_control,TE_knockdown,log2_FC_TE
20589,ENSMUSG00000077711.3,125,14,0,0,2,0.066667,3.000000,5.491853
29134,ENSMUSG00000049555.11,3094,19,2,0,3,0.050000,1.333333,4.736966
46221,ENSMUSG00000085705.2,1155,15,0,1,2,0.125000,3.000000,4.584963
8910,ENSMUSG00000097807.2,1818,11,0,0,1,0.083333,2.000000,4.584963
52242,ENSMUSG00000081118.2,1055,11,0,0,1,0.083333,2.000000,4.584963


In [8]:
!pip install mygene

  Using cached mygene-3.2.2-py2.py3-none-any.whl.metadata (10 kB)
Using cached mygene-3.2.2-py2.py3-none-any.whl (5.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mygene]

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import pandas as pd
import numpy as np
import mygene

# 1. 데이터 불러오기 및 TE 계산 (고정된 경로 사용)
df = pd.read_csv('binfo1-work/read-counts.txt', sep='\t', comment='#')
cols_to_keep = ['Geneid', 'Length', 'RNA-siLuc.bam', 'RNA-siLin28a.bam', 'RPF-siLuc.bam', 'RPF-siLin28a.bam']
df_clean = df[cols_to_keep].copy()
df_clean = df_clean[df_clean['RNA-siLuc.bam'] >= 10]

df_clean['TE_control'] = (df_clean['RPF-siLuc.bam'] + 1) / (df_clean['RNA-siLuc.bam'] + 1)
df_clean['TE_knockdown'] = (df_clean['RPF-siLin28a.bam'] + 1) / (df_clean['RNA-siLin28a.bam'] + 1)
df_clean['log2_FC_TE'] = np.log2(df_clean['TE_knockdown'] / df_clean['TE_control'])

# ---------------------------------------------------------
# 2. Ensembl ID 번역기 가동 (이번엔 유전자 풀네임 'name'도 함께 가져옵니다!)
df_clean['ensembl_id'] = df_clean['Geneid'].apply(lambda x: x.split('.')[0])

print("NCBI 데이터베이스에서 유전자 이름과 상세 기능을 조회 중입니다...")
mg = mygene.MyGeneInfo()
# fields에 'name'을 추가하여 유전자의 전체 기능 설명을 확보합니다.
gene_info = mg.querymany(df_clean['ensembl_id'], scopes='ensembl.gene', fields='symbol,name', species='mouse', as_dataframe=True)

gene_info = gene_info.reset_index()[['query', 'symbol', 'name']].drop_duplicates(subset='query')
master_df = pd.merge(df_clean, gene_info, left_on='ensembl_id', right_on='query', how='left')

# ---------------------------------------------------------
# 3. 키워드 기반의 철저한 면역(Immune) 유전자 필터링 시스템
# 유전자 이름(Symbol)이나 전체 설명(Name)에 아래 단어가 포함되어 있으면 면역 유전자로 인정!
immune_keywords = ['immune', 'histocompatibility', 'antigen', 'interleukin', 'chemokine', 't-cell', 'b-cell', 'lymphocyte']

def check_immune(row):
    symbol = str(row['symbol']).lower() if pd.notnull(row['symbol']) else ""
    name = str(row['name']).lower() if pd.notnull(row['name']) else ""
    
    # 설정한 키워드가 유전자 이름이나 설명에 하나라도 들어가있는지 확인
    return any(kw in symbol or kw in name for kw in immune_keywords)

master_df['is_immune'] = master_df.apply(check_immune, axis=1)

# ---------------------------------------------------------
# 4. 결과 확인
total_immune = master_df['is_immune'].sum()
print(f"\n✅ 필터링 완료! 총 {total_immune}개의 면역 관련 유전자를 찾아냈습니다.")

print("\n🔥 완벽해진 기준으로 뽑은, Lin28a Knockdown 시 번역이 떡상한 면역 유전자 Top 5 🔥")
display(master_df[master_df['is_immune']].sort_values('log2_FC_TE', ascending=False)[['symbol', 'name', 'log2_FC_TE']].head())

NCBI 데이터베이스에서 유전자 이름과 상세 기능을 조회 중입니다...


1 input query terms found dup hits:	[('ENSMUSG00000072694', 2)]
47 input query terms found no hit:	['ENSMUSG00000103821', 'ENSMUSG00000101599', 'ENSMUSG00000067017', 'ENSMUSG00000104340', 'ENSMUSG000



✅ 필터링 완료! 총 157개의 면역 관련 유전자를 찾아냈습니다.

🔥 완벽해진 기준으로 뽑은, Lin28a Knockdown 시 번역이 떡상한 면역 유전자 Top 5 🔥


,symbol,name,log2_FC_TE
5630,Il11,interleukin 11,2.115477
13676,H2-Bl,"histocompatibility 2, blastocyst",1.935460
826,Cd84,CD84 antigen,1.652077
2634,Cd2,CD2 antigen,1.543793
7466,Il34,interleukin 34,1.432959


In [11]:
!pip install gseapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.2/633.2 kB 2.6 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 2.9 MB/s  0:00:12m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gseapy]

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [12]:
import pandas as pd
import numpy as np
import gseapy as gp

# (앞부분 데이터 로드 및 TE 계산 코드는 동일하다고 가정합니다)
# master_df에 유전자 'symbol'이 들어있는 상태에서 시작합니다.

print("🌐 세계 공인 면역 유전자 데이터베이스(MSigDB)에서 목록을 가져오는 중입니다...")

# MSigDB의 C7(Immunologic signature, 면역학 표준 세트) 또는 GO(Gene Ontology) 데이터를 가져옵니다.
# 여기서는 가장 표준적인 GO Biological Process에서 면역 유전자 세트를 가져옵니다.
try:
    # 생쥐(Mouse)의 유전자 세트 라이브러리 로드
    gene_sets = gp.get_library_name(organism='Mouse')
    
    # 'GO_Biological_Process_2023' 라이브러리에서 면역 관련 공식 유전자 목록 추출
    go_bp = gp.get_library('GO_Biological_Process_2023', organism='Mouse')
    
    # 'immune response (GO:0006955)'에 속하는 공식 유전자들만 쏙 뽑아내기
    official_immune_genes = []
    for term, genes in go_bp.items():
        if 'immune response' in term.lower():
            official_immune_genes.extend(genes)
            
    # 중복 제거 및 대문자/소문자 통일 (생쥐는 보통 첫글자만 대문자)
    official_immune_genes = list(set([g.capitalize() for g in official_immune_genes]))
    
    # ---------------------------------------------------------
    # 3. 진짜 논문 방식으로 라벨링하기
    # 우리 데이터의 유전자 이름이 공식 면역 유전자 목록에 있으면 True, 없으면 False!
    master_df['is_immune'] = master_df['symbol'].str.capitalize().isin(official_immune_genes)
    
    print(f"✅ 검증 완료! 공인 데이터베이스 기준으로 총 {master_df['is_immune'].sum()}개의 진짜 면역 유전자가 마크되었습니다.")

except Exception as e:
    print(f"네트워크 오류 또는 라이브러리 문제 발생: {e}")
    print("임시로 이전 키워드 방식을 유지합니다.")

🌐 세계 공인 면역 유전자 데이터베이스(MSigDB)에서 목록을 가져오는 중입니다...
✅ 검증 완료! 공인 데이터베이스 기준으로 총 301개의 진짜 면역 유전자가 마크되었습니다.


In [13]:
# 1. 서열 분석을 위한 Biopython 라이브러리 설치
!pip install biopython

# 2. Ensembl 데이터베이스에서 생쥐의 전체 CDS(단백질 번역 서열) 압축 파일 다운로드 (약 15MB, 금방 받아집니다!)
!wget http://ftp.ensembl.org/pub/release-110/fasta/mus_musculus/cds/Mus_musculus.GRCm39.cds.all.fa.gz -O mouse_cds.fa.gz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 3.9 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
--2026-05-21 17:37:28--  http://ftp.ensembl.org/pub/release-110/fasta/mus_musculus/cds/Mus_musculus.GRCm39.cds.all.fa.gz
Resolving ftp.ensembl.org (ftp.ensembl.org)... 193.62.193.169
Connecting to ftp.ensembl.org (ftp.ensembl.org)|193.62.193.169|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17195690 (16M) [application/x-gzip]
Saving to: ‘mouse_cds.fa.gz’

mouse_cds.fa.gz     100%[===================>]  16.40M   430KB/s    in 26s     

2026-05-21 17:37:58 (637 KB/s) - ‘mouse_cds.fa.gz’ saved [17195690/17195690]



In [14]:
import pandas as pd
import gzip
from Bio import SeqIO
from Bio.SeqUtils import gc_fraction

print("🧬 생쥐 전체 mRNA 서열에서 암호를 해독하는 중입니다...")

sequence_features = []

# 1. 압축된 FASTA 서열 파일을 엽니다.
with gzip.open('mouse_cds.fa.gz', 'rt') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        # 헤더에서 Ensembl Gene ID 추출 (예: gene:ENSMUSG00000012345.6 -> ENSMUSG00000012345)
        # FASTA 파일의 설명(description) 부분을 쪼개서 유전자 ID를 찾습니다.
        description = record.description
        gene_id_part = [x for x in description.split() if x.startswith('gene:')]
        
        if gene_id_part:
            raw_gene_id = gene_id_part[0].split(':')[1]
            ensembl_id = raw_gene_id.split('.')[0] # 버전 번호 제거
            
            # 2. 분석할 서열 데이터 (문자열로 변환)
            seq = str(record.seq).upper()
            
            # 3. 핵심 Feature 계산!
            # - GC 비율 (높을수록 RNA 구조가 단단하게 접힘)
            # - GGAG 모티프 개수 (LIN28A가 들러붙는 표적 서열)
            gc_content = gc_fraction(seq) * 100
            ggag_count = seq.count('GGAG')
            seq_length = len(seq)
            
            sequence_features.append({
                'ensembl_id': ensembl_id,
                'gc_content': gc_content,
                'ggag_count': ggag_count,
                'cds_length': seq_length
            })

# 데이터프레임으로 변환 (동일한 유전자의 여러 transcript가 있을 경우 평균값 사용)
seq_df = pd.DataFrame(sequence_features)
seq_df = seq_df.groupby('ensembl_id').mean().reset_index()

print("✅ 서열 암호 해독 완료!")
display(seq_df.head())

🧬 생쥐 전체 mRNA 서열에서 암호를 해독하는 중입니다...
✅ 서열 암호 해독 완료!


,ensembl_id,gc_content,ggag_count,cds_length
0,ENSMUSG00000000001,43.755869,11.000,1065.000000
1,ENSMUSG00000000003,41.732229,4.000,469.500000
2,ENSMUSG00000000028,49.850633,9.000,1224.666667
3,ENSMUSG00000000037,45.432895,20.625,2166.000000
4,ENSMUSG00000000049,53.196847,2.750,438.750000


In [ ]:
# 기존 master_df와 서열 Feature 데이터프레임(seq_df)을 ensembl_id 기준으로 병합!
final_master_df = pd.merge(master_df, seq_df, on='ensembl_id', how='inner')

# 머신러닝에 불필요한 결측치 제거
final_master_df = final_master_df.dropna(subset=['gc_content', 'ggag_count'])

# 최종 데이터 저장 (이 파일을 깃허브에 올리면 됩니다!)
final_master_df.to_csv('processed_master_table.csv', index=False)

print(f"🎉 축하합니다! 총 {len(final_master_df)}개 유전자의 [발현량 + 면역 여부 + 서열 특징] 통합 테이블이 완성되었습니다.")

# 최종 테이블 구경하기
cols_to_show = ['symbol', 'is_immune', 'log2_FC_TE', 'gc_content', 'ggag_count', 'cds_length']
display(final_master_df[cols_to_show].head(10))

🎉 축하합니다! 총 12703개 유전자의 [발현량 + 면역 여부 + 서열 특징] 통합 테이블이 완성되었습니다.


,symbol,is_immune,log2_FC_TE,gc_content,ggag_count,cds_length
0,Mrpl15,False,-1.016743,52.352148,5.000000,542.250000
1,Lypla1,False,-0.384426,47.682977,3.142857,536.857143
2,Tcea1,False,-0.827799,42.053685,4.500000,922.500000
3,Atp6v1h,False,-1.191550,43.215896,4.400000,965.400000
4,Oprk1,False,0.169925,49.010787,8.000000,1164.750000
5,Rb1cc1,False,-0.542688,41.100653,7.250000,1872.875000
6,Pcmtd1,False,0.177652,43.139101,3.000000,498.333333
7,Rrs1,False,-1.233445,64.116576,20.000000,1098.000000
8,Adhfe1,False,-0.668794,49.835884,3.000000,642.600000
9,Mybl1,False,-0.222288,40.995825,5.333333,1711.000000


In [17]:
# 1. 테이블 전체에서 True인 면역 유전자가 총 몇 개나 숨어있는지 개수 확인
true_count = final_master_df['is_immune'].sum()
print(f"📊 현재 데이터셋에 존재하는 면역 유전자(True) 총 개수: {true_count}개 / 전체 {len(final_master_df)}개")

# 2. is_immune 컬럼이 True인 행들만 쏙 필터링해서 보여주기
immune_genes_df = final_master_df[final_master_df['is_immune'] == True]

# 3. 면역 유전자들 중 번역 효율(log2_FC_TE)이 높은 순서대로 10개 구경하기
cols_to_show = ['symbol', 'is_immune', 'log2_FC_TE', 'gc_content', 'ggag_count', 'cds_length']
display(immune_genes_df[cols_to_show].sort_values('log2_FC_TE', ascending=False).head(10))

📊 현재 데이터셋에 존재하는 면역 유전자(True) 총 개수: 301개 / 전체 12703개


,symbol,is_immune,log2_FC_TE,gc_content,ggag_count,cds_length
9993,C6,True,2.000000,45.954848,17.500000,2052.25
402,Cd55,True,1.741383,45.401739,4.500000,883.50
9172,Hfe,True,1.579013,53.911730,6.500000,630.00
2823,Trim62,True,1.440573,63.768179,8.000000,933.00
1118,Serping1,True,0.856471,52.659516,6.000000,1279.50
12087,Ifit2,True,0.807355,45.130549,5.500000,772.00
8295,Skap1,True,0.736966,50.810171,5.833333,707.00
109,Il1r1,True,0.567889,43.353674,13.500000,1726.50
5394,Spn,True,0.566914,57.854342,6.000000,933.00
7968,Vamp2,True,0.543332,60.763724,3.666667,325.00
